# 02 Basket Analysis

This notebook reconstructs the basket/co-purchase analysis in a portable, memory-first form. It measures which analysis categories appear within the same order and exports thesis-ready basket tables.


## Pipeline position

This notebook follows `notebooks/01_data_preparation_master_panel.ipynb` and precedes `notebooks/03_regression_analysis.ipynb`. Because basket analysis requires order-level co-purchase information, it reads `data/Dataset.parquet` directly and applies the same filtering and category-construction logic used for the shared transaction universe.


## Inputs

The required input is `data/Dataset.parquet`. Aggregated outputs from notebook 01 are not sufficient for this analysis because they do not preserve `OrdreID`.


## Outputs

When executed, the notebook writes the basket pair store, CSV tables, HTML tables, and run checks under `outputs/basket_analysis/`:

- `outputs/basket_analysis/basket_order_category_pairs.sqlite`
- `outputs/basket_analysis/basket_support.csv`
- `outputs/basket_analysis/basket_support.html`
- `outputs/basket_analysis/food_drink_summary.csv`
- `outputs/basket_analysis/food_drink_summary.html`
- `outputs/basket_analysis/pair_table_main.csv`
- `outputs/basket_analysis/pair_table_main.html`
- `outputs/basket_analysis/focus_pairs.csv`
- `outputs/basket_analysis/focus_pairs.html`
- `outputs/basket_analysis/focus_pairs_a4.html`
- `outputs/basket_analysis/basket_analysis_checks.csv`

The SQLite file is a disk-backed intermediate used to avoid holding all order-category pairs in memory.


## Methodological notes

The basket unit is the order, identified by `OrdreID`. The analysis categories are `Dinner`, `Lunch`, `Sweets`, `Salads`, `Dessert`, `Cold Drinks`, and `Hot Drinks`. `Ekskluderes` is retained in the shared category mapping but excluded from basket tables. Co-purchase means that two analysis categories appear in the same order, and `lift > 1` indicates positive association rather than a causal effect.

The transaction file contains approximately 16-20 million rows, so the implementation uses selected columns, chunked Parquet reads where available, disk-backed pair storage, and explicit cleanup.


## Environment and portable paths

The notebook is intended for the `csvi_env` kernel. Paths are discovered from the project root and are not hardcoded to a local machine path.


In [ ]:
import gc
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

MARKER_FILES = ('README_FOR_CODEX.md', 'AGENTS.md', 'pyproject.toml')


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if any((candidate / marker).exists() for marker in MARKER_FILES):
            return candidate
    raise FileNotFoundError(
        'Could not find the project root. Start Jupyter from inside the project folder '
        'or make sure README_FOR_CODEX.md, AGENTS.md, or pyproject.toml exists.'
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_BASKET_DIR = OUTPUT_DIR / 'basket_analysis'

DATASET_PATH = DATA_DIR / 'Dataset.parquet'

PAIRS_DB_PATH = OUTPUT_BASKET_DIR / 'basket_order_category_pairs.sqlite'
SUPPORT_CSV_PATH = OUTPUT_BASKET_DIR / 'basket_support.csv'
SUPPORT_HTML_PATH = OUTPUT_BASKET_DIR / 'basket_support.html'
FOOD_DRINK_CSV_PATH = OUTPUT_BASKET_DIR / 'food_drink_summary.csv'
FOOD_DRINK_HTML_PATH = OUTPUT_BASKET_DIR / 'food_drink_summary.html'
PAIR_TABLE_MAIN_CSV_PATH = OUTPUT_BASKET_DIR / 'pair_table_main.csv'
PAIR_TABLE_MAIN_HTML_PATH = OUTPUT_BASKET_DIR / 'pair_table_main.html'
FOCUS_PAIRS_CSV_PATH = OUTPUT_BASKET_DIR / 'focus_pairs.csv'
FOCUS_PAIRS_HTML_PATH = OUTPUT_BASKET_DIR / 'focus_pairs.html'
FOCUS_PAIRS_A4_HTML_PATH = OUTPUT_BASKET_DIR / 'focus_pairs_a4.html'
CHECKS_PATH = OUTPUT_BASKET_DIR / 'basket_analysis_checks.csv'

OUTPUT_BASKET_DIR.mkdir(parents=True, exist_ok=True)


def project_relative(path):
    return path.relative_to(PROJECT_ROOT).as_posix()


def require_file(path, description):
    if not path.exists():
        raise FileNotFoundError(
            f'Missing {description}: {project_relative(path)}. '
            'Place the file under the documented project folder before running this notebook.'
        )


def require_columns(frame, required_columns, frame_name):
    missing = [column for column in required_columns if column not in frame.columns]
    if missing:
        raise KeyError(f'{frame_name} is missing required columns: {missing}')


def report_memory(frame, name):
    memory_gb = frame.memory_usage(deep=True).sum() / 1_000_000_000
    print(f'{name}: shape={frame.shape}, approx_memory={memory_gb:.3f} GB')
    return memory_gb


print(f'Project root: {PROJECT_ROOT}')
print(f'Basket output folder: {project_relative(OUTPUT_BASKET_DIR)}')

## Input checks

The next cell verifies file existence and Parquet schema metadata without loading the full transaction dataset.


In [ ]:
REQUIRED_BASKET_COLUMNS = [
    'OrdreID',
    'ArtikkelNavn',
    'HovedartikkelgruppeTxt',
    'Artikkelkategori1Txt',
    'Produktgruppe',
    'OrdrekildeTekst',
]


def parquet_column_status(path, required_columns):
    try:
        import pyarrow.parquet as pq

        parquet_file = pq.ParquetFile(path)
        available_columns = set(parquet_file.schema_arrow.names)
        metadata = {
            'file': project_relative(path),
            'exists': path.exists(),
            'size_mb': round(path.stat().st_size / 1_000_000, 2),
            'row_groups': parquet_file.num_row_groups,
            'metadata_rows': parquet_file.metadata.num_rows,
        }
        columns = pd.DataFrame(
            {
                'column': required_columns,
                'available': [column in available_columns for column in required_columns],
            }
        )
        return metadata, columns
    except Exception as exc:
        metadata = {
            'file': project_relative(path),
            'exists': path.exists(),
            'size_mb': round(path.stat().st_size / 1_000_000, 2) if path.exists() else np.nan,
            'row_groups': pd.NA,
            'metadata_rows': pd.NA,
            'note': f'Schema inspection failed: {exc}',
        }
        columns = pd.DataFrame({'column': required_columns, 'available': pd.NA})
        return metadata, columns


require_file(DATASET_PATH, 'transaction source data')
dataset_metadata, basket_column_status = parquet_column_status(DATASET_PATH, REQUIRED_BASKET_COLUMNS)

display(pd.DataFrame([dataset_metadata]))
display(basket_column_status)

## Basket settings and category rules

These settings define the analysis categories, food/drink blocks, focus pairs, and main-pair co-order threshold used in the basket tables.


In [ ]:
CATS = ['Dinner', 'Lunch', 'Sweets', 'Salads', 'Dessert', 'Cold Drinks', 'Hot Drinks']
FOOD_CATS = {'Dinner', 'Lunch', 'Sweets', 'Salads', 'Dessert'}
DRINK_CATS = {'Cold Drinks', 'Hot Drinks'}

FOCUS_PAIRS = [
    ('Cold Drinks', 'Dinner'),
    ('Cold Drinks', 'Salads'),
    ('Hot Drinks', 'Sweets'),
    ('Hot Drinks', 'Lunch'),
]

MIN_CO_ORDERS = 10_000

EXCLUDED_MAIN_GROUPS = ['Non-food']
EXCLUDED_CATEGORIES = ['Kioskvarer', 'Catering', 'Tilbehør', 'Alkoholdrikk']
EXCLUDED_PRODUCT_GROUPS = ['Vin', 'Øl', 'Tilbehør']

FILTER_TEXT_COLUMNS = [
    'HovedartikkelgruppeTxt',
    'Artikkelkategori1Txt',
    'Produktgruppe',
    'OrdrekildeTekst',
]

print(f'Basket categories: {CATS}')
print(f'Main pair table threshold: co_orders >= {MIN_CO_ORDERS:,}')

## Disk-backed pair-store helpers

The helper functions implement the original pair definition through a SQLite-backed store. This avoids constructing the full order-category pair table in memory.


In [ ]:
def bool_array(mask):
    if isinstance(mask, pd.Series):
        return mask.fillna(False).to_numpy(dtype=bool)
    return np.asarray(mask, dtype=bool)


def optimize_string_columns(frame, columns, max_category_share=0.50):
    n_rows = len(frame)
    if n_rows == 0:
        return frame
    for column in columns:
        if column in frame.columns and frame[column].dtype == 'object':
            unique_share = frame[column].nunique(dropna=False) / n_rows
            if unique_share <= max_category_share:
                frame[column] = frame[column].astype('category')
    return frame


def prepare_basket_pairs(raw_chunk):
    """Return unique order-category pairs from one chunk using the Hovedfilen category logic."""
    require_columns(raw_chunk, REQUIRED_BASKET_COLUMNS, 'raw_chunk')
    raw_chunk = optimize_string_columns(raw_chunk, FILTER_TEXT_COLUMNS)

    keep_mask = np.ones(len(raw_chunk), dtype=bool)
    keep_mask &= ~bool_array(raw_chunk['HovedartikkelgruppeTxt'].isin(EXCLUDED_MAIN_GROUPS))
    keep_mask &= ~bool_array(raw_chunk['Artikkelkategori1Txt'].isin(EXCLUDED_CATEGORIES))
    keep_mask &= ~bool_array(raw_chunk['Produktgruppe'].isin(EXCLUDED_PRODUCT_GROUPS))
    keep_mask &= bool_array(raw_chunk['OrdrekildeTekst'] != 'Netthandel')
    keep_mask &= bool_array(raw_chunk['ArtikkelNavn'] != 'Diverse brød')
    keep_mask &= bool_array(raw_chunk['Artikkelkategori1Txt'] != 'Småretter')

    # Chunk-sized shallow copy avoids copying the full transaction dataset.
    filtered = raw_chunk.loc[keep_mask, ['OrdreID', 'ArtikkelNavn', 'Artikkelkategori1Txt']].copy(deep=False)
    del keep_mask

    missing_order_ids = int(filtered['OrdreID'].isna().sum())
    if missing_order_ids:
        raise ValueError(
            f'Found {missing_order_ids:,} missing OrdreID values in a basket chunk. '
            'Basket analysis requires order identifiers.'
        )

    article_name = filtered['ArtikkelNavn'].astype('string')
    category_1 = filtered['Artikkelkategori1Txt']

    category_conditions = [
        article_name.str.contains('Club sandwich', case=False, na=False)
        | category_1.isin(['Burger', 'Middag']),
        article_name.str.contains('Focaccia', case=False, na=False) | (category_1 == 'Brødmat'),
        category_1.isin(['Bakst', 'Kaker']),
        category_1 == 'Salat',
        category_1 == 'Dessert',
        category_1 == 'Kald drikke',
        category_1 == 'Varm drikke',
    ]

    analysis_values = np.select(category_conditions, CATS, default='Ekskluderes')
    pair_frame = pd.DataFrame(
        {
            'order_id': filtered['OrdreID'].astype('string'),
            'Analyse_Kategori': analysis_values,
        }
    )
    pair_frame = pair_frame.loc[pair_frame['Analyse_Kategori'].isin(CATS)].drop_duplicates()
    pair_frame['Analyse_Kategori'] = pair_frame['Analyse_Kategori'].astype('category')

    stats = {
        'raw_rows': len(raw_chunk),
        'filtered_rows': len(filtered),
        'chunk_unique_pairs': len(pair_frame),
        'missing_order_ids': missing_order_ids,
    }

    del filtered, article_name, category_1, analysis_values
    gc.collect()
    return pair_frame, stats


def open_pair_store(path):
    if 'basket_conn' in globals():
        try:
            basket_conn.close()
        except Exception:
            pass
    if path.exists():
        path.unlink()
    conn = sqlite3.connect(path)
    conn.execute('PRAGMA journal_mode=WAL')
    conn.execute('PRAGMA synchronous=NORMAL')
    conn.execute(
        'CREATE TABLE order_category_pairs ('
        'order_id TEXT NOT NULL, '
        'Analyse_Kategori TEXT NOT NULL, '
        'PRIMARY KEY (order_id, Analyse_Kategori)) WITHOUT ROWID'
    )
    return conn


def insert_pair_frame(conn, pair_frame):
    if pair_frame.empty:
        return 0
    records = pair_frame[['order_id', 'Analyse_Kategori']].itertuples(index=False, name=None)
    conn.executemany(
        'INSERT OR IGNORE INTO order_category_pairs (order_id, Analyse_Kategori) VALUES (?, ?)',
        records,
    )
    return len(pair_frame)


def count_pair_store_rows(conn):
    return int(conn.execute('SELECT COUNT(*) FROM order_category_pairs').fetchone()[0])

## Build order-category pair store

This heavy preparation step reads only the required transaction columns and processes them in chunks when `pyarrow.dataset` is available. It should be run only when basket outputs need to be rebuilt.


In [ ]:
def build_pair_store_from_dataset(path, batch_size=500_000):
    conn = open_pair_store(PAIRS_DB_PATH)
    stats = {
        'read_strategy': 'pyarrow.dataset chunked',
        'raw_rows': 0,
        'filtered_rows': 0,
        'chunk_unique_pairs_before_global_dedup': 0,
        'missing_order_ids': 0,
        'batches': 0,
    }

    try:
        import pyarrow.dataset as ds

        dataset = ds.dataset(path, format='parquet')
        available_columns = set(dataset.schema.names)
        missing_columns = [column for column in REQUIRED_BASKET_COLUMNS if column not in available_columns]
        if missing_columns:
            raise KeyError(f'Transaction Parquet is missing required columns: {missing_columns}')

        scanner = dataset.scanner(columns=REQUIRED_BASKET_COLUMNS, batch_size=batch_size)
        for batch_number, record_batch in enumerate(scanner.to_batches(), start=1):
            raw_chunk = record_batch.to_pandas()
            pair_frame, chunk_stats = prepare_basket_pairs(raw_chunk)
            insert_pair_frame(conn, pair_frame)
            conn.commit()

            stats['batches'] = batch_number
            stats['raw_rows'] += chunk_stats['raw_rows']
            stats['filtered_rows'] += chunk_stats['filtered_rows']
            stats['chunk_unique_pairs_before_global_dedup'] += chunk_stats['chunk_unique_pairs']
            stats['missing_order_ids'] += chunk_stats['missing_order_ids']

            del raw_chunk, pair_frame, record_batch
            gc.collect()

            if batch_number % 10 == 0:
                print(
                    f'Processed {batch_number:,} batches; '
                    f'raw rows={stats["raw_rows"]:,}; '
                    f'filtered rows={stats["filtered_rows"]:,}; '
                    f'global pairs={count_pair_store_rows(conn):,}'
                )

    except ImportError:
        stats['read_strategy'] = 'pandas selected-column fallback'
        print('pyarrow.dataset is unavailable; using pandas selected-column fallback.')
        raw_chunk = pd.read_parquet(path, columns=REQUIRED_BASKET_COLUMNS)
        report_memory(raw_chunk, 'selected transaction columns before basket filtering')
        pair_frame, chunk_stats = prepare_basket_pairs(raw_chunk)
        insert_pair_frame(conn, pair_frame)
        conn.commit()

        stats['batches'] = 1
        stats['raw_rows'] = chunk_stats['raw_rows']
        stats['filtered_rows'] = chunk_stats['filtered_rows']
        stats['chunk_unique_pairs_before_global_dedup'] = chunk_stats['chunk_unique_pairs']
        stats['missing_order_ids'] = chunk_stats['missing_order_ids']

        del raw_chunk, pair_frame
        gc.collect()

    stats['global_unique_order_category_pairs'] = count_pair_store_rows(conn)
    return conn, stats


basket_conn, pair_store_stats = build_pair_store_from_dataset(DATASET_PATH)

print(f'Read strategy: {pair_store_stats["read_strategy"]}')
print(f'Raw rows processed: {pair_store_stats["raw_rows"]:,}')
print(f'Rows after current filters: {pair_store_stats["filtered_rows"]:,}')
print(f'Global unique order-category pairs: {pair_store_stats["global_unique_order_category_pairs"]:,}')
print(f'Saved pair store: {project_relative(PAIRS_DB_PATH)}')

## Support, lift, and food/drink summaries

This step computes support, lift, conditional probabilities, the main pair table, focus pairs, and food/drink summaries from the disk-backed pair store.


In [ ]:
def compute_support_from_store(conn):
    n_orders = int(conn.execute('SELECT COUNT(DISTINCT order_id) FROM order_category_pairs').fetchone()[0])
    support_counts = pd.read_sql_query(
        'SELECT Analyse_Kategori, COUNT(*) AS category_orders '
        'FROM order_category_pairs GROUP BY Analyse_Kategori',
        conn,
    )
    support_counts = support_counts.set_index('Analyse_Kategori').reindex(CATS, fill_value=0)
    support = (support_counts['category_orders'] / n_orders).astype(float)
    support_df = support.rename('support').reset_index()
    support_df.columns = ['Analyse_Kategori', 'support']
    support_df = support_df.sort_values('support', ascending=False).reset_index(drop=True)
    return n_orders, support, support_df


def compute_pair_table_from_store(conn, support, n_orders):
    pair_counts = pd.read_sql_query(
        'SELECT a.Analyse_Kategori AS A, b.Analyse_Kategori AS B, COUNT(*) AS co_orders '
        'FROM order_category_pairs a '
        'JOIN order_category_pairs b ON a.order_id = b.order_id '
        'WHERE a.Analyse_Kategori < b.Analyse_Kategori '
        'GROUP BY a.Analyse_Kategori, b.Analyse_Kategori',
        conn,
    )
    if pair_counts.empty:
        return pd.DataFrame(
            columns=[
                'A',
                'B',
                'co_orders',
                'p_ab',
                'lift',
                'p_b_given_a',
                'p_a_given_b',
                'support_A',
                'support_B',
            ]
        )

    pair_counts['p_ab'] = pair_counts['co_orders'] / n_orders
    p_a = pair_counts['A'].map(support).astype(float)
    p_b = pair_counts['B'].map(support).astype(float)
    pair_counts['lift'] = pair_counts['p_ab'] / (p_a * p_b)
    pair_counts['p_b_given_a'] = pair_counts['p_ab'] / p_a
    pair_counts['p_a_given_b'] = pair_counts['p_ab'] / p_b
    pair_counts['support_A'] = p_a
    pair_counts['support_B'] = p_b
    pair_counts = pair_counts.sort_values(
        ['lift', 'co_orders'],
        ascending=[False, False],
    ).reset_index(drop=True)
    return pair_counts


def placeholders(values):
    return '(' + ','.join(['?'] * len(values)) + ')'


def compute_food_drink_summary_from_store(
    conn,
    n_orders,
    food_cats=FOOD_CATS,
    drink_cats=DRINK_CATS,
):
    food_values = list(food_cats)
    drink_values = list(drink_cats)
    food_sql = placeholders(food_values)
    drink_sql = placeholders(drink_values)

    n_food = int(
        conn.execute(
            f'SELECT COUNT(DISTINCT order_id) FROM order_category_pairs '
            f'WHERE Analyse_Kategori IN {food_sql}',
            food_values,
        ).fetchone()[0]
    )
    n_drink = int(
        conn.execute(
            f'SELECT COUNT(DISTINCT order_id) FROM order_category_pairs '
            f'WHERE Analyse_Kategori IN {drink_sql}',
            drink_values,
        ).fetchone()[0]
    )
    n_both = int(
        conn.execute(
            f'SELECT COUNT(DISTINCT f.order_id) '
            f'FROM order_category_pairs f '
            f'JOIN order_category_pairs d ON f.order_id = d.order_id '
            f'WHERE f.Analyse_Kategori IN {food_sql} AND d.Analyse_Kategori IN {drink_sql}',
            food_values + drink_values,
        ).fetchone()[0]
    )

    p_food = n_food / n_orders
    p_drink = n_drink / n_orders
    p_both = n_both / n_orders
    p_drink_given_food = p_both / p_food if p_food > 0 else np.nan
    p_food_given_drink = p_both / p_drink if p_drink > 0 else np.nan
    lift_food_drink = p_both / (p_food * p_drink) if (p_food > 0 and p_drink > 0) else np.nan

    return pd.DataFrame(
        [
            {
                'n_orders': int(n_orders),
                'n_food_orders': int(n_food),
                'n_drink_orders': int(n_drink),
                'n_both_orders': int(n_both),
                'P(Food)': float(p_food),
                'P(Drink)': float(p_drink),
                'P(Both)': float(p_both),
                'P(Drink|Food)': float(p_drink_given_food),
                'P(Food|Drink)': float(p_food_given_drink),
                'Lift(Food,Drink)': float(lift_food_drink),
            }
        ]
    )


def extract_focus_pairs(pair_table, focus_pairs=FOCUS_PAIRS):
    order_map = {tuple(sorted(pair)): index for index, pair in enumerate(focus_pairs)}
    tmp = pair_table.copy()
    tmp['pair_key'] = tmp.apply(lambda row: tuple(sorted([str(row['A']), str(row['B'])])), axis=1)
    out = tmp[tmp['pair_key'].isin(list(order_map.keys()))].copy()
    out['order_idx'] = out['pair_key'].map(order_map)
    out = out.sort_values('order_idx').drop(columns=['pair_key', 'order_idx']).reset_index(drop=True)
    return out


n_orders, support, support_df = compute_support_from_store(basket_conn)
pair_table = compute_pair_table_from_store(basket_conn, support, n_orders)
pair_table_main = pair_table[pair_table['co_orders'] >= MIN_CO_ORDERS].copy()
food_drink_summary = compute_food_drink_summary_from_store(basket_conn, n_orders)
focus_table = extract_focus_pairs(pair_table, focus_pairs=FOCUS_PAIRS)

print(f'Unique orders (n_orders): {n_orders:,}')
print(f'Unique order-category pairs: {pair_store_stats["global_unique_order_category_pairs"]:,}')
display(support_df)
display(food_drink_summary)
display(pair_table_main)
display(focus_table)

## Formatting helpers

The formatting helpers apply the established thesis table style and route rendered tables to the basket-analysis output folder.


In [ ]:
def format_support_table(support_df, cat_col='Analyse_Kategori'):
    out = support_df.copy()
    out = out.rename(columns={cat_col: 'Category', 'support': 'support'})
    out['support'] = out['support'].map(lambda value: f'{value:.6f}')
    return out


def format_food_drink_summary(summary_df):
    out = summary_df.copy()
    int_cols = ['n_orders', 'n_food_orders', 'n_drink_orders', 'n_both_orders']
    for column in int_cols:
        out[column] = out[column].map(lambda value: f'{int(value):,}')
    float_cols = [
        'P(Food)',
        'P(Drink)',
        'P(Both)',
        'P(Drink|Food)',
        'P(Food|Drink)',
        'Lift(Food,Drink)',
    ]
    for column in float_cols:
        out[column] = out[column].map(lambda value: f'{value:.6f}')
    return out


def format_pair_table(frame):
    out = frame.copy()
    out = out[['A', 'B', 'co_orders', 'p_ab', 'lift', 'p_b_given_a', 'p_a_given_b']].copy()
    out['co_orders'] = out['co_orders'].map(lambda value: f'{int(value):,}')
    out['p_ab'] = out['p_ab'].map(lambda value: f'{value:.6f}')
    out['lift'] = out['lift'].map(lambda value: f'{value:.3f}')
    out['p_b_given_a'] = out['p_b_given_a'].map(lambda value: f'{value:.3f}')
    out['p_a_given_b'] = out['p_a_given_b'].map(lambda value: f'{value:.3f}')
    return out


def save_html_table(frame, out_path, title, note=''):
    css = '''
<style>
  body { font-family: "Times New Roman", Times, serif; font-size: 10pt; }
  .title { text-align: center; font-size: 12pt; font-weight: bold; margin: 0 0 8px 0; }
  .note { margin-top: 8px; font-size: 9.5pt; }
  table { border-collapse: collapse; width: 100%; border-top: 1.2px solid #000; border-bottom: 1.2px solid #000; }
  th, td { border: none; padding: 3px 5px; }
  thead tr:last-child th { border-bottom: 1px solid #000; }
  th { text-align: center; }
  td { text-align: right; }
  td:first-child, td:nth-child(2) { text-align: left; }
</style>
'''.strip()
    html = f'''
<html>
  <head>{css}</head>
  <body>
    <div class="title">{title}</div>
    {frame.to_html(index=False, escape=False)}
    <div class="note">{note}</div>
  </body>
</html>
'''.strip()
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(html, encoding='utf-8')
    return out_path


def export_focus_pairs_a4(focus_df, out_path, title, note):
    df = format_pair_table(focus_df)
    css = '''
<style>
  @page { size: A4 landscape; margin: 12mm; }
  html, body { width: 297mm; margin: 0; padding: 0; }
  body { font-family: "Times New Roman", Times, serif; font-size: 9.5pt; }
  .title { text-align: center; font-size: 12pt; font-weight: bold; margin: 0 0 6px 0; }
  .note { margin-top: 6px; font-size: 9.5pt; text-align: left; }
  table { border-collapse: collapse; width: 100%; table-layout: fixed; border-top: 1.2px solid #000; border-bottom: 1.2px solid #000; margin: 0; }
  th, td { border: none; padding: 2px 4px; vertical-align: top; }
  thead th { text-align: center; font-weight: bold; padding-bottom: 4px; line-height: 1.05; }
  thead tr:last-child th { border-bottom: 1px solid #000; }
  th:nth-child(1), td:nth-child(1) { width: 16%; text-align: left; white-space: nowrap; }
  th:nth-child(2), td:nth-child(2) { width: 16%; text-align: left; white-space: nowrap; }
  th:nth-child(3), td:nth-child(3) { width: 14%; text-align: right; white-space: nowrap; }
  th:nth-child(4), td:nth-child(4) { width: 14%; text-align: right; white-space: nowrap; }
  th:nth-child(5), td:nth-child(5) { width: 12%; text-align: right; white-space: nowrap; }
  th:nth-child(6), td:nth-child(6) { width: 14%; text-align: right; white-space: nowrap; }
  th:nth-child(7), td:nth-child(7) { width: 14%; text-align: right; white-space: nowrap; }
</style>
'''.strip()
    html = f'''
<html>
  <head>{css}</head>
  <body>
    <div class="title">{title}</div>
    {df.to_html(index=False, escape=False)}
    <div class="note">{note}</div>
  </body>
</html>
'''.strip()
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(html, encoding='utf-8')
    return out_path

## Export basket outputs

This step writes CSV and HTML versions of each basket table, including the A4 focus-pairs table. The mirrored outputs make the basket results inspectable and traceable.


In [ ]:
support_pretty = format_support_table(support_df, cat_col='Analyse_Kategori')
food_drink_pretty = format_food_drink_summary(food_drink_summary)
pair_main_pretty = format_pair_table(pair_table_main)
focus_pretty = format_pair_table(focus_table)

support_df.to_csv(SUPPORT_CSV_PATH, index=False)
food_drink_summary.to_csv(FOOD_DRINK_CSV_PATH, index=False)
pair_table_main.to_csv(PAIR_TABLE_MAIN_CSV_PATH, index=False)
focus_table.to_csv(FOCUS_PAIRS_CSV_PATH, index=False)

save_html_table(
    support_pretty,
    SUPPORT_HTML_PATH,
    title='Basket support: P(category in order)',
    note='Notes: support is the share of unique orders containing each analysis category.',
)
save_html_table(
    food_drink_pretty,
    FOOD_DRINK_HTML_PATH,
    title='Food and drink basket summary',
    note='Notes: probabilities are computed at the order level.',
)
save_html_table(
    pair_main_pretty,
    PAIR_TABLE_MAIN_HTML_PATH,
    title=f'Basket pair table: co_orders >= {MIN_CO_ORDERS:,}',
    note='Notes: lift > 1 indicates positive association between categories within the same order.',
)
save_html_table(
    focus_pretty,
    FOCUS_PAIRS_HTML_PATH,
    title='Focus pairs: drinks and key food categories',
    note='Notes: co_orders is the number of orders where both categories appear.',
)
export_focus_pairs_a4(
    focus_table,
    FOCUS_PAIRS_A4_HTML_PATH,
    title='Table 2 - Focus pairs: drinks and key food categories',
    note=(
        'Notes: co_orders is the number of orders where both categories appear. '
        'lift > 1 indicates positive association.'
    ),
)

output_files = [
    SUPPORT_CSV_PATH,
    SUPPORT_HTML_PATH,
    FOOD_DRINK_CSV_PATH,
    FOOD_DRINK_HTML_PATH,
    PAIR_TABLE_MAIN_CSV_PATH,
    PAIR_TABLE_MAIN_HTML_PATH,
    FOCUS_PAIRS_CSV_PATH,
    FOCUS_PAIRS_HTML_PATH,
    FOCUS_PAIRS_A4_HTML_PATH,
]

for path in output_files:
    print(f'Saved: {project_relative(path)}')

## Basket checks

The final checks record the basket run, key row counts, and output locations in a small traceability table.


In [ ]:
check_rows = []


def add_check(name, value, note=''):
    check_rows.append({'check': name, 'value': value, 'note': note})


add_check(
    'read_strategy',
    pair_store_stats['read_strategy'],
    'Transaction read strategy used for basket pair construction',
)
add_check('raw_rows_processed', pair_store_stats['raw_rows'], 'Rows read from selected transaction columns')
add_check('rows_after_filters', pair_store_stats['filtered_rows'], 'Rows after Hovedfilen exclusion filters')
add_check(
    'chunk_unique_pairs_before_global_dedup',
    pair_store_stats['chunk_unique_pairs_before_global_dedup'],
    'Chunk-level unique pairs before SQLite global de-duplication',
)
add_check(
    'global_unique_order_category_pairs',
    pair_store_stats['global_unique_order_category_pairs'],
    'Rows in disk-backed unique pair store',
)
add_check('n_orders', n_orders, 'Unique orders with at least one included basket category')
add_check('n_pair_table_rows', len(pair_table), 'All unordered category pairs')
add_check('n_pair_table_main_rows', len(pair_table_main), f'Pairs with co_orders >= {MIN_CO_ORDERS:,}')
add_check('n_focus_pairs', len(focus_table), 'Focus-pair rows found')

for path in [PAIRS_DB_PATH, *output_files]:
    add_check(f'output_exists_{path.name}', path.exists(), project_relative(path))

checks_df = pd.DataFrame(check_rows)
checks_df.to_csv(CHECKS_PATH, index=False)

print(f'Saved checks: {project_relative(CHECKS_PATH)}')
display(checks_df)

## Downstream use

After execution, basket tables are available under `outputs/basket_analysis/`. The disk-backed pair store may be inspected or removed later if disk space matters. Continue with `notebooks/03_regression_analysis.ipynb`. This notebook is descriptive basket analysis; it does not modify the realised-weather master panel and does not create forecast-weather features for ML.
